In [3]:
import pandas as pd
df = pd.read_csv("../data/processed/features_reviews.csv")

In [4]:
df['label_num']= df['label'].map({'CG': 1, 'OR':0})

In [5]:
from sklearn.model_selection import train_test_split
X_train_idx, X_test_idx = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=df['label_num']
)
train_df = df.loc[X_train_idx]
test_df = df.loc[X_test_idx]

In [6]:
train_df['label_num'].value_counts(normalize=True)
test_df['label_num'].value_counts(normalize=True)

label_num
0    0.500062
1    0.499938
Name: proportion, dtype: float64

In [7]:
df['clean_text'] = df['clean_text'].fillna("")

In [8]:
df['clean_text'].isnull().sum()


np.int64(0)

In [9]:
df['label_num'] = df['label'].map({'CG': 1, 'OR': 0})

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(train_df['clean_text'])
X_test_tfidf = tfidf.transform(test_df['clean_text'])

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

ValueError: np.nan is an invalid document, expected byte or unicode string.

In [11]:
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

NameError: name 'X_train_tfidf' is not defined

In [21]:
pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [22]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

X_train_embeddings = model.encode(train_df['text_'].tolist(), show_progress_bar=True)
X_test_embeddings = model.encode(test_df['text_'].tolist(), show_progress_bar=True)

OSError: [WinError 4551] An Application Control policy has blocked this file. Error loading "C:\Users\acer\FakeGuard-AI-Review-Detector\venv\Lib\site-packages\torch\lib\torch_python.dll" or one of its dependencies.

In [12]:
print(X_train_embeddings.shape)
print(X_test_embeddings.shape)

(32345, 384)
(8087, 384)


In [13]:
import numpy as np
X_train_embeddings = np.load('../data/processed/X_train_embeddings.npy')
X_test_embeddings = np.load('../data/processed/X_test_embeddings.npy')

In [14]:
print(X_train_tfidf.shape, X_train_embeddings.shape)
print(X_test_tfidf.shape, X_test_embeddings.shape)

NameError: name 'X_train_tfidf' is not defined

In [15]:

# Cell 1: Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [16]:
# Cell 2: Load your feature-engineered data
df = pd.read_csv('../data/processed/features_reviews.csv')

In [17]:
# Cell 3: Fix the NaN issue (empty cleaned reviews) — must redo every session
df['clean_text'] = df['clean_text'].fillna("")
df['clean_text'].isnull().sum()  # should print 0

np.int64(0)

In [18]:
# Cell 4: Map labels to numbers
df['label_num'] = df['label'].map({'CG': 1, 'OR': 0})

In [19]:
# Cell 5: Train-test split (same random_state=42 as Colab, so splits match)
X_train_idx, X_test_idx = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=df['label_num']
)
train_df = df.loc[X_train_idx]
test_df = df.loc[X_test_idx]

In [20]:
# Cell 6: Build TF-IDF features
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(train_df['clean_text'])
X_test_tfidf = tfidf.transform(test_df['clean_text'])

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(32345, 5000)
(8087, 5000)


In [21]:
# Cell 7: Load Sentence-BERT embeddings generated on Colab
X_train_embeddings = np.load('../data/processed/X_train_embeddings.npy')
X_test_embeddings = np.load('../data/processed/X_test_embeddings.npy')

print(X_train_embeddings.shape)
print(X_test_embeddings.shape)

(32345, 384)
(8087, 384)


In [22]:
# Cell 8: Naive Bayes on TF-IDF
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, train_df['label_num'])
nb_preds = nb_model.predict(X_test_tfidf)

print("Naive Bayes (TF-IDF)")
print(classification_report(test_df['label_num'], nb_preds))

Naive Bayes (TF-IDF)
              precision    recall  f1-score   support

           0       0.85      0.88      0.87      4044
           1       0.88      0.85      0.86      4043

    accuracy                           0.86      8087
   macro avg       0.87      0.86      0.86      8087
weighted avg       0.87      0.86      0.86      8087



In [23]:
# Cell 9: Logistic Regression on TF-IDF
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, train_df['label_num'])
lr_tfidf_preds = lr_tfidf.predict(X_test_tfidf)

print("Logistic Regression (TF-IDF)")
print(classification_report(test_df['label_num'], lr_tfidf_preds))

Logistic Regression (TF-IDF)
              precision    recall  f1-score   support

           0       0.89      0.89      0.89      4044
           1       0.89      0.89      0.89      4043

    accuracy                           0.89      8087
   macro avg       0.89      0.89      0.89      8087
weighted avg       0.89      0.89      0.89      8087



In [24]:
# Cell 10: Logistic Regression on Sentence-BERT embeddings
lr_embed = LogisticRegression(max_iter=1000)
lr_embed.fit(X_train_embeddings, train_df['label_num'])
lr_embed_preds = lr_embed.predict(X_test_embeddings)

print("Logistic Regression (Sentence-BERT)")
print(classification_report(test_df['label_num'], lr_embed_preds))

Logistic Regression (Sentence-BERT)
              precision    recall  f1-score   support

           0       0.81      0.81      0.81      4044
           1       0.81      0.81      0.81      4043

    accuracy                           0.81      8087
   macro avg       0.81      0.81      0.81      8087
weighted avg       0.81      0.81      0.81      8087



In [25]:
import joblib

joblib.dump(lr_tfidf, '../models/lr_tfidf_model.pkl')
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')

['../models/tfidf_vectorizer.pkl']

In [27]:
import re
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

In [28]:
loaded_model = joblib.load('../models/lr_tfidf_model.pkl')
loaded_vectorizer = joblib.load('../models/tfidf_vectorizer.pkl')

sample_review = "This product is absolutely amazing! Best purchase ever! Highly recommend!"
sample_clean = clean_text(sample_review)  # reuse your Day 2 cleaning function
sample_vec = loaded_vectorizer.transform([sample_clean])
prediction = loaded_model.predict(sample_vec)
probability = loaded_model.predict_proba(sample_vec)

print("Prediction:", "Fake" if prediction[0] == 1 else "Real")
print("Confidence:", probability[0])

Prediction: Real
Confidence: [0.72519954 0.27480046]


In [29]:
sample_idx = test_df[test_df['label_num'] == 1].index[0]  # grab a known fake review
real_sample_text = df.loc[sample_idx, 'text_']
real_sample_clean = df.loc[sample_idx, 'clean_text']

print("Actual label: Fake")
print("Review:", real_sample_text)

sample_vec = loaded_vectorizer.transform([real_sample_clean])
prediction = loaded_model.predict(sample_vec)
probability = loaded_model.predict_proba(sample_vec)

print("Predicted:", "Fake" if prediction[0] == 1 else "Real")
print("Confidence:", probability[0])

Actual label: Fake
Review: My grandson loves to experience the real world with the trains.  We also have the Clipper.
Predicted: Fake
Confidence: [0.38652185 0.61347815]
